# Molecular Docking Pipeline

This notebook provides a plug-and-play pipeline for protein preparation, ligand preparation, and molecular docking using AutoDock Vina.

### Prerequisites
Ensure you have the environment set up using the provided `environment.yml` file.

In [ ]:
from docking_utils import Preprocessor, VinaDocking, get_box_from_ligand

In [ ]:
# 1. Setup paths
protein_pdb = 'path/to/your/protein.pdb'
ligand_file = 'path/to/your/ligand.sdf'  # or .pdb

output_dir = 'results'
os.makedirs(output_dir, exist_ok=True)

receptor_pdbqt = os.path.join(output_dir, 'receptor.pdbqt')
ligand_pdbqt = os.path.join(output_dir, 'ligand.pdbqt')
docking_output = os.path.join(output_dir, 'docking_poses.pdbqt')

## Step 1: Prepare Protein and Ligand

In [ ]:
# Initialize preprocessor (set fix_protein=True to use PDBFixer)
prep = Preprocessor(fix_protein=True)

# Prepare receptor (Add keep_water=True if you need to keep structural waters)
prep.prepare_receptor(protein_pdb, receptor_pdbqt, keep_water=False)

# Prepare ligand
prep.prepare_ligand(ligand_file, ligand_pdbqt.replace('.pdbqt', '.sdf'), in_format='sdf')

# Note: Convert ligand to PDBQT using meeko if not already done
os.system(f'mk_prepare_ligand.py -i {ligand_pdbqt.replace(".pdbqt", ".sdf")} -o {ligand_pdbqt}')

## Step 2: Define Docking Box and Run Docking

In [ ]:
# Define docking box based on a native ligand (if available) or manual coordinates
center, size = get_box_from_ligand(ligand_file, padding=5.0)
print(f"Box Center: {center}\nBox Size: {size}")

# Initialize and run Vina
docker = VinaDocking(receptor_pdbqt, center, size, num_poses=9, exhaustiveness=8)
results = docker.dock(ligand_pdbqt, docking_output)

print("Docking Results:")
print(results.head())

## Step 3: Re-scoring with MM-GBSA

In [ ]:
from docking_utils import MMGBSACalculator
from rdkit import Chem
from rdkit.Chem import AllChem
import os

# Initialize MM-GBSA calculator
mmgbsa = MMGBSACalculator(implicit_solvent='OBC2')

# Extract docking poses and score each
mmgbsa_results = []

# Convert docking output to individual SDF files for scoring
poses_sdf = os.path.join(output_dir, 'poses.sdf')

# Read docking poses (PDBQT) and convert to SDF for each pose
pose_suppl = Chem.SDMolSupplier(poses_sdf) if os.path.exists(poses_sdf) else None

if pose_suppl:
    for i, mol in enumerate(pose_suppl):
        if mol is None:
            continue
        # Save individual pose
        pose_sdf = os.path.join(output_dir, f'pose_{i}.sdf')
        writer = Chem.SDWriter(pose_sdf)
        writer.write(mol)
        writer.close()
        
        # Calculate MM-GBSA score
        try:
            score = mmgbsa.calculate_score(protein_pdb, pose_sdf)
            mmgbsa_results.append({'pose': i, 'mmgbsa_score': score})
            print(f"Pose {i}: MM-GBSA = {score:.2f} kcal/mol")
        except Exception as e:
            print(f"Error scoring pose {i}: {e}")
        
        # Clean up individual personality file
        os.remove(pose_sdf)

print("\nMM-GBSA Results:")
print(pd.DataFrame(mmgbsa_results))